# Simulacion de cartera de inversion optima

In [1]:
# Instalar librerías (solo la primera vez)
!pip -q install yfinance scipy --quiet

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Activos de tu cartera (tickers de Yahoo Finance)
activos = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'JPM', 'JNJ', 'KO']

# Índices contra los cuales comparar
indices = {
    'S&P 500': '^GSPC',
    'Nasdaq 100': '^NDX',
    'Dow Jones': '^DJI',
    'Russell 2000': '^RUT'
}

# Período de análisis
fecha_inicio = '2020-01-01'
fecha_fin = '2025-12-31'

# Tasa libre de riesgo anual (bonos del tesoro USA ~4.5%)
tasa_libre_riesgo = 0.045

# Cantidad de portafolios a simular
n_portafolios = 10000

print(f"Activos seleccionados: {activos}")
print(f"Período: {fecha_inicio} a {fecha_fin}")

Activos seleccionados: ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'JPM', 'JNJ', 'KO']
Período: 2020-01-01 a 2025-12-31


In [3]:
print("Descargando datos de activos...")
data = yf.download(activos, start=fecha_inicio, end=fecha_fin, auto_adjust=True)['Close']
data = data.dropna()

print("Descargando datos de índices...")
data_indices = yf.download(list(indices.values()), start=fecha_inicio, end=fecha_fin, auto_adjust=True)['Close']
data_indices.columns = list(indices.keys())
data_indices = data_indices.dropna()

# Calcular retornos diarios
retornos = data.pct_change().dropna()
retornos_indices = data_indices.pct_change().dropna()

# Estadísticas anualizadas (252 días hábiles)
retorno_medio = retornos.mean() * 252
matriz_cov = retornos.cov() * 252

print(f"\n✅ Datos descargados: {len(data)} días")
print(f"\nRetornos anuales esperados de cada activo:")
print((retorno_medio * 100).round(2).astype(str) + '%')

Descargando datos de activos...


[*********************100%***********************]  8 of 8 completed
[                       0%                       ]

Descargando datos de índices...


[*********************100%***********************]  4 of 4 completed


✅ Datos descargados: 1507 días

Retornos anuales esperados de cada activo:
Ticker
AAPL     27.25%
AMZN     21.37%
GOOGL     30.9%
JNJ      10.59%
JPM      21.57%
KO         9.2%
MSFT     23.81%
NVDA     71.79%
dtype: object


In [4]:
def estadisticas_portafolio(pesos, retornos_medios, cov_matrix, rf):
    """Calcula retorno, volatilidad y Sharpe del portafolio"""
    retorno = np.sum(retornos_medios * pesos)
    volatilidad = np.sqrt(np.dot(pesos.T, np.dot(cov_matrix, pesos)))
    sharpe = (retorno - rf) / volatilidad
    return retorno, volatilidad, sharpe

def neg_sharpe(pesos, retornos_medios, cov_matrix, rf):
    return -estadisticas_portafolio(pesos, retornos_medios, cov_matrix, rf)[2]

def min_volatilidad(pesos, cov_matrix):
    return np.sqrt(np.dot(pesos.T, np.dot(cov_matrix, pesos)))

def optimizar_sharpe(retornos_medios, cov_matrix, rf):
    n = len(retornos_medios)
    args = (retornos_medios, cov_matrix, rf)
    restricciones = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    limites = tuple((0, 1) for _ in range(n))
    pesos_iniciales = np.array([1/n] * n)
    resultado = minimize(neg_sharpe, pesos_iniciales, args=args,
                        method='SLSQP', bounds=limites, constraints=restricciones)
    return resultado.x

def optimizar_min_vol(cov_matrix):
    n = cov_matrix.shape[0]
    restricciones = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    limites = tuple((0, 1) for _ in range(n))
    pesos_iniciales = np.array([1/n] * n)
    resultado = minimize(min_volatilidad, pesos_iniciales, args=(cov_matrix,),
                        method='SLSQP', bounds=limites, constraints=restricciones)
    return resultado.x

print("✅ Funciones de optimización definidas")

✅ Funciones de optimización definidas


In [5]:
# Cartera de máximo Sharpe Ratio
pesos_sharpe = optimizar_sharpe(retorno_medio, matriz_cov, tasa_libre_riesgo)
ret_sharpe, vol_sharpe, sr_sharpe = estadisticas_portafolio(pesos_sharpe, retorno_medio, matriz_cov, tasa_libre_riesgo)

# Cartera de mínima volatilidad
pesos_minvol = optimizar_min_vol(matriz_cov)
ret_minvol, vol_minvol, sr_minvol = estadisticas_portafolio(pesos_minvol, retorno_medio, matriz_cov, tasa_libre_riesgo)

# Cartera equiponderada (referencia)
pesos_eq = np.array([1/len(activos)] * len(activos))
ret_eq, vol_eq, sr_eq = estadisticas_portafolio(pesos_eq, retorno_medio, matriz_cov, tasa_libre_riesgo)

print("="*60)
print("CARTERA ÓPTIMA (Máximo Sharpe Ratio)")
print("="*60)
df_sharpe = pd.DataFrame({'Activo': activos, 'Peso (%)': pesos_sharpe * 100})
df_sharpe = df_sharpe[df_sharpe['Peso (%)'] > 0.01].sort_values('Peso (%)', ascending=False)
print(df_sharpe.to_string(index=False))
print(f"\nRetorno anual: {ret_sharpe*100:.2f}% | Volatilidad: {vol_sharpe*100:.2f}% | Sharpe: {sr_sharpe:.3f}")

print("\n" + "="*60)
print("CARTERA DE MÍNIMA VOLATILIDAD")
print("="*60)
df_minvol = pd.DataFrame({'Activo': activos, 'Peso (%)': pesos_minvol * 100})
df_minvol = df_minvol[df_minvol['Peso (%)'] > 0.01].sort_values('Peso (%)', ascending=False)
print(df_minvol.to_string(index=False))
print(f"\nRetorno anual: {ret_minvol*100:.2f}% | Volatilidad: {vol_minvol*100:.2f}% | Sharpe: {sr_minvol:.3f}")

CARTERA ÓPTIMA (Máximo Sharpe Ratio)
Activo  Peso (%)
    KO 64.539250
  AMZN 26.124354
 GOOGL  5.447018
  NVDA  3.889379

Retorno anual: 51.62% | Volatilidad: 36.72% | Sharpe: 1.283

CARTERA DE MÍNIMA VOLATILIDAD
Activo  Peso (%)
  AMZN 46.056980
   JPM 36.852580
  MSFT 12.928477
 GOOGL  3.857511
  NVDA  0.304451

Retorno anual: 12.29% | Volatilidad: 16.71% | Sharpe: 0.466
